### 1. Import Libraries and Load Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import nltk

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Load IMDb dataset
import kagglehub

# Download the latest version of IMDb dataset
dataset_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
df = pd.read_csv(f"{dataset_path}/IMDB Dataset.csv")

# Display the first few rows of the dataset
print(df.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive




### 2. Data Preprocessing



In [ ]:
from bs4 import BeautifulSoup

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Define stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()
    # Replace non-alphanumeric characters with spaces
    text = re.sub(r'\W+', ' ', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenization
    tokens = word_tokenize(text)
    # Lemmatization and stopword removal
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess_text)

# Display the first few rows
print(df[['review', 'cleaned_review']].head())


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one reviewer mentioned watching oz episode hoo...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically family little boy jake think zombie ...  
4  petter mattei love time money visually stunnin...  




### 3. Dataset Annotation with Emotions



In [ ]:
!pip install tqdm

In [ ]:
from transformers import pipeline, AutoTokenizer
from tqdm import tqdm  # Import tqdm for progress bar

# Load the model and tokenizer
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
emotion_classifier = pipeline(
    "text-classification",
    model=model_name,
    tokenizer=tokenizer,
    top_k=None  # Replaces return_all_scores=True
)

def get_emotions(text):
    try:
        emotions = emotion_classifier(text, truncation=True, max_length=512)
        return {emotion['label']: emotion['score'] for emotion in emotions[0]}
    except Exception as e:
        print(f"Error processing text: {e}")
        return {label: 0.0 for label in ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']}

# Apply the emotion analysis with progress bar
tqdm.pandas(desc="Processing reviews")
df['emotions'] = df['cleaned_review'].progress_apply(get_emotions)

# Display the first few rows of the dataset with emotion annotations
print(df[['cleaned_review', 'emotions']].head())

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu
Processing reviews: 100%|██████████| 50000/50000 [3:57:45<00:00,  3.51it/s]


                                      cleaned_review  \
0  one reviewer mentioned watching oz episode hoo...   
1  wonderful little production filming technique ...   
2  thought wonderful way spend time hot summer we...   
3  basically family little boy jake think zombie ...   
4  petter mattei love time money visually stunnin...   

                                            emotions  
0  {'fear': 0.9398821592330933, 'anger': 0.028225...  
1  {'joy': 0.9506715536117554, 'fear': 0.02154896...  
2  {'joy': 0.9554997086524963, 'surprise': 0.0173...  
3  {'sadness': 0.5102293491363525, 'surprise': 0....  
4  {'joy': 0.9587359428405762, 'surprise': 0.0266...  


In [ ]:
# Save imdb with emotions
# Select and reorder columns from existing df
selected_columns = ['cleaned_review', 'sentiment', 'emotions']
df = df[selected_columns]

# Save df with selected columns
df.to_csv('imdb_with_emotions.csv', index=False)

# Display first rows to verify
print(df.head())

                                      cleaned_review sentiment  \
0  one reviewer mentioned watching oz episode hoo...  positive   
1  wonderful little production filming technique ...  positive   
2  thought wonderful way spend time hot summer we...  positive   
3  basically family little boy jake think zombie ...  negative   
4  petter mattei love time money visually stunnin...  positive   

                                            emotions  
0  {'fear': 0.9398821592330933, 'anger': 0.028225...  
1  {'joy': 0.9506715536117554, 'fear': 0.02154896...  
2  {'joy': 0.9554997086524963, 'surprise': 0.0173...  
3  {'sadness': 0.5102293491363525, 'surprise': 0....  
4  {'joy': 0.9587359428405762, 'surprise': 0.0266...  
